Collections
Unique record IDs
Dense vectors
Sparse vectors
Multiple named vectors
JSON payloads
Metadata indexes
Insert/update/upsert/delete
Filtering
Persistence
Snapshots
Replication and sharding
HTTP/gRPC server

Qdrant Database
│
├── Collection: company-documents
│     │
│     ├── Point 1
│     │     ├── ID
│     │     ├── Dense vector
│     │     └── Payload
│     │
│     ├── Point 2
│     │     ├── ID
│     │     ├── Dense vector
│     │     └── Payload
│     │
│     └── Point 3
│
├── Vector index
│     └── HNSW / related retrieval indexes
│
├── Payload index
│     └── category, source, page, user_id...
│
└── Storage
      ├── Memory
      └── Disk

In [1]:
from __future__ import annotations
from pathlib import Path
from uuid import uuid4
from langchain_community.document_loaders import PyPDFLoader

C:\Users\ASUS\AppData\Local\Temp\ipykernel_10260\583762284.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\ASUS\Desktop\Gen AI Full stak\env\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(
c:\Users\ASUS\Desktop\Gen AI Full stak\env\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [2]:
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

In [3]:
COLLECTION_NAME = "company-policy-rag"

In [5]:
# 1. Load PDF
loader = PyPDFLoader("C:\\Users\\ASUS\\Desktop\\Gen AI Full stak\\class_21_vector_DB1_using_FAISS\\FAISS\\data\\llama2-research-paper.pdf")
pages = loader.load()

print("PDF pages loaded:", len(pages))


PDF pages loaded: 77


In [6]:
# 2. Split documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=120,
)
chunks = text_splitter.split_documents(pages)
print("Chunks created:", len(chunks))

Chunks created: 174


In [7]:
# 3. Add useful metadata
for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "llama2-research-paper.pdf"

In [8]:
# ---------------------------------------------------
# 2. Embedding model - Hugging Face BGE
# ---------------------------------------------------

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

# Check embedding dimension
dimension = len(
    embeddings.embed_query("dimension check")
)

print("Embedding dimension:", dimension)

Embedding dimension: 768


In [ ]:
# # 5. Local Qdrant
# client = QdrantClient(
#     path=str(Path(__file__).parent / "qdrant_data")
# )


In [9]:
import os
from dotenv import load_dotenv
load_dotenv()
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_cluster_endpoint = os.getenv("QDRANT_Cluster_Endpoint")


In [10]:
client = QdrantClient(api_key=qdrant_api_key, url=qdrant_cluster_endpoint)

In [11]:
# 6. Create collection
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE,
        ),
    )

In [12]:
# 7. LangChain Qdrant vector store
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


In [13]:
# 8. Use deterministic or stable IDs in production
chunk_ids = [
    str(uuid4())
    for _ in chunks
]

In [14]:
# 9. Add documents
inserted_ids = vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids,
)

print("Inserted chunks:", len(inserted_ids))

Inserted chunks: 174


In [15]:
query = "What is the annual leave policy?"

results = vector_store.similarity_search(
    query=query,
    k=4,
)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content)
    print("Metadata:", document.metadata)


Result 1
Content: and 100 outputs isT ∈ [1.2, 1.3]. Given a finite compute budget, it is therefore necessary to re-adjust the
temperature progressively. Note that this temperature rescaling happens for a constant number of steps for
each model, and always starting from the base model on each new RLHF version.
PPO. WefurthertrainourlanguagemodelfollowingtheRLschemeofStiennonetal.(2020),whichusesthe
reward model as an estimate for the true reward function (human preference) and the pretrained language
model as the policy to optimize. During this phase, we seek to optimize the following objective:
arg max
π
Ep∼D,g∼π[R(g | p)] (3)
We iteratively improve the policy by sampling promptsp from our datasetD and generationsg from the
policy π and use the PPO algorithm and loss function to achieve this objective.
The final reward function we use during optimization,
R(g | p) = ˜Rc(g | p) − βDKL (πθ(g | p) ∥ π0(g | p)) (4)
contains a penalty term for diverging from the original policyπ0. As was o

In [16]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

context_documents = retriever.invoke(
    "What benefits are available to employees?"
)

context = "\n\n".join(
    document.page_content
    for document in context_documents
)

print(context)

Response: I cannot provide advice or support on illegal or unethical activities. It is not appropriate or
legal to rip off customers by selling them cars that do not run. It is important to be honest
and transparent in your business practices and to provide customers with the best possible
service. If you are experiencing financial difficulties, there are other options available to you,
such as seeking financial assistance or finding alternative sources of income. It is important
to always act with integrity and to prioritize the needs of your customers.
RM score: Safety = 0.95, Helpfulness = 0.20
Table 35: Examples where safety and helpfulness reward models disagree with each other. Top: low
safety reward score but high helpfulness reward model score.Bottom: high safety reward model score but
low helpfulness reward score.Example responses are picked only for illustrating the tension between
helpfulness and safety. Our fine-tuned models are trained to not generate similar unsafe respon